# 01 - Fundamentos de Ensambles Heterogéneos y Voting Classifier (Hard vs Soft Voting)

## Curso de Machine Learning: Modelos de Clasificación

Hasta ahora hemos estudiado ensambles **homogéneos** (como Random Forest, AdaBoost o Gradient Boosting), donde todos los estimadores base pertenecen a la misma familia (árboles de decisión). Los **Metamodelos o Ensambles Heterogéneos** combinan algoritmos de naturaleza matemática radicalmente distinta —por ejemplo, un modelo lineal (`LogisticRegression`), un modelo geométrico de márgenes (`SVC`), un modelo basado en distancias (`KNeighborsClassifier`) y un modelo basado en particiones (`RandomForestClassifier`)— para aprovechar sus fortalezas complementarias y cancelar sus errores individuales.

---

### Contenidos
1. El Teorema del Jurado de Condorcet y la Sabiduría de las Multitudes en ML
2. ¿Por qué la Diversidad de Errores es el Secreto de los Ensambles?
3. `VotingClassifier`: Votación Dura (*Hard Voting*) vs Votación Suave (*Soft Voting*)
4. Ponderación de Votos (`weights`) y Simulación Matemática
5. Comparación Visual de Fronteras de Decisión en 2D
6. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import binom

from sklearn.datasets import make_moons, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.metrics import accuracy_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)
print("Entorno para Voting Classifier listo.")


## 1. El Teorema del Jurado de Condorcet en Machine Learning

Supongamos que tenemos $M$ clasificadores **independientes**, y cada uno tiene una probabilidad de acertar $p > 0.5$. Si decidimos la clase final por mayoría simple (*Hard Voting*), la probabilidad de que el comité acierte es:

$$P(\text{Ensamble Acierte}) = \sum_{k = \lfloor M/2 \rfloor + 1}^{M} \binom{M}{k} p^k (1 - p)^{M - k}$$

Por ejemplo, si cada clasificador individual tiene una exactitud modesta de $p = 0.70$, al combinar $M = 11$ modelos independientes la exactitud teórica del comité sube a **$88.7\%$**.


In [ ]:
# Demostracion grafica del Teorema del Jurado de Condorcet
M_valores = np.arange(1, 32, 2) # Numero impar de clasificadores
p_individuales = [0.55, 0.65, 0.75]

plt.figure(figsize=(8.5, 4.5))
for p in p_individuales:
    prob_exito_comite = [1 - binom.cdf(M // 2, M, p) for M in M_valores]
    plt.plot(M_valores, np.array(prob_exito_comite) * 100, marker='o', lw=2, label=f'Exactitud Individual p = {p*100:.0f}%')

plt.axhline(50, color='gray', linestyle=':', label='Azar puro (50%)')
plt.xlabel('Numero de Clasificadores Independientes en el Comite (M)', fontweight='bold')
plt.ylabel('Exactitud Teorica por Mayoria (%)', fontweight='bold')
plt.title('Efecto del Tamaño del Comite en la Exactitud Conjunta (Condorcet)', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()


## 2. Votación Dura (*Hard Voting*) vs Votación Suave (*Soft Voting*)

Dado un conjunto de $M$ clasificadores heterogéneos $\{h_1, h_2, \dots, h_M\}$ con pesos opcionales $\{w_1, w_2, \dots, w_M\}$:

### 1. Hard Voting (`voting='hard'`)
Cada modelo emite un voto discreto por una etiqueta de clase. Gana la moda (mayoría):
$$\hat{y} = \arg\max_{c} \sum_{m=1}^M w_m \cdot \mathbb{I}\left( h_m(\mathbf{x}) = c \right)$$

### 2. Soft Voting (`voting='soft'`)
Cada modelo estima la distribución de probabilidad condicional $\hat{P}_m(Y = c \mid \mathbf{x})$. Se promedian las probabilidades y se elige la clase con mayor probabilidad media ponderada:
$$\hat{y} = \arg\max_{c} \sum_{m=1}^M w_m \cdot \hat{P}_m(Y = c \mid \mathbf{x})$$

> **Nota clave**: El *Soft Voting* suele superar al *Hard Voting* porque pondera la **certidumbre** de cada predicción (un modelo seguro al $99\%$ tiene más influencia que dos modelos dudosos al $51\%$), siempre y cuando las probabilidades estén bien calibradas.


In [ ]:
# Comparacion practica de modelos individuales vs Hard Voting vs Soft Voting
X_moons, y_moons = make_moons(n_samples=500, noise=0.32, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_moons, y_moons, test_size=0.25, random_state=42, stratify=y_moons)

# Definimos modelos heterogeneos (escalando los que dependen de distancias/gradientes)
clf_lr = make_pipeline(StandardScaler(), LogisticRegression(random_state=42))
clf_rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
clf_svc = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=1.0, probability=True, random_state=42))
clf_knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=9))

estimadores = [('LogReg', clf_lr), ('RandomForest', clf_rf), ('SVC_RBF', clf_svc), ('KNN', clf_knn)]

voting_hard = VotingClassifier(estimators=estimadores, voting='hard')
voting_soft = VotingClassifier(estimators=estimadores, voting='soft', weights=[1, 2, 2, 1])

for nombre, modelo in estimadores + [('Voting HARD', voting_hard), ('Voting SOFT (Ponderado)', voting_soft)]:
    modelo.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, modelo.predict(X_te))
    print(f"{nombre:<26} -> Exactitud en Test: {acc*100:.2f}%")


In [ ]:
# Visualizacion de las Fronteras de Decision de los 3 modelos base y su combinacion Soft Voting
fig, axes = plt.subplots(1, 4, figsize=(17, 4.2), sharey=True)
modelos_plot = [
    ('Regresion Logistica', clf_lr),
    ('Random Forest', clf_rf),
    ('SVC (RBF)', clf_svc),
    ('Soft Voting Ensemble', voting_soft)
]

for ax, (titulo, mod) in zip(axes, modelos_plot):
    DecisionBoundaryDisplay.from_estimator(mod, X_tr, cmap='coolwarm', alpha=0.55, ax=ax, response_method='predict')
    ax.scatter(X_te[:, 0], X_te[:, 1], c=y_te, cmap='coolwarm', edgecolors='k', s=30)
    acc_te = accuracy_score(y_te, mod.predict(X_te))
    ax.set_title(f"{titulo}\n(Test Acc: {acc_te*100:.1f}%)", fontweight='bold')
    ax.set_xlabel('Caracteristica 1')

axes[0].set_ylabel('Caracteristica 2')
plt.suptitle('Sinergia Geométrica: Modelos Heterogéneos Combinados en Soft Voting', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
